In [1]:
#import packages
import time
import numpy as np
import capytaine as cpt
import scipy
from capytaine.io.mesh_writers import write_STL
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.linalg import eigh
import vtk
import logging
import xarray as xr
from capytaine.io.xarray import merge_complex_values
from capytaine.post_pro import rao
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s: %(message)s')
# user defined functions
import DM_ShowNodes as DMshow
import DM_Reading as dm_r
import DM_Assemble as DM_A
import SEREP as SEREP
import DM_Method as DM_M
import RODM_complex_interconnection as R_cx

In [4]:
hydrodynamic_data_path = f"E:\phd\Code\DM-FEM2D\HydrodynamicData\Yoon_hinge\DM10_10_direction0_wl180.nc"
dataset = merge_complex_values(xr.open_dataset(hydrodynamic_data_path))

### 引入铰接模型，100*100的交接模型（用于验证）12个铰接，30*30的模块铰接，10*10的模块交接模型
### 结构材料参数：rho 281.875 , E = 4.53e8
### 实现的基本步骤：1、建立水动力数据。2、建立结构矩阵数据。3、识别出铰接节点位置。4、

In [ ]:
def input_parameters():
    # 输入文件路径
    file_m = r'E:\phd\Code\DM-FEM2D\StructureData\Hinge_complex_paper4\Job3030hinge-1_MASS1.mtx'
    file_k = r'E:\phd\Code\DM-FEM2D\StructureData\Hinge_complex_paper4\Job3030hinge-1_STIF1.mtx'
    hydro_file = r'E:\phd\Code\DM-FEM2D\HydrodynamicData\Yoon_hinge\DM10_10_direction0_wl180.nc'  # 水动力数据文件路径

    # 获取刚度矩阵和质量矩阵
    m = dm_r.get_stiffness_matrix(file_m)
    k = dm_r.get_stiffness_matrix(file_k)

    # 设置结构的数量
    n = 100  # 例如，3个结构

    master_nodes = generate_control_points()
    print("主要控制点：", master_nodes)


    # 使用相同的函数生成刚度矩阵和质量矩阵的块对角矩阵
    fem_k = create_block_matrix(k, n)
    fem_m = create_block_matrix(m, n)

    # 铰链刚度
    k_hinge =1e10
    
    # 定义所有铰接节点对的列表
    hinge_x_pairs = R_cx.print_hinge_x_pairs(grid_size=10, N=49, nodes_per_row=7, total_rows=7)
    hinge_y_pairs = R_cx.print_hinge_y_pairs(grid_size=10, N=49, nodes_per_row=7, total_rows=7)
    print(hinge_x_pairs)

    stiffness_x = R_cx.apply_hinge_joints(N=49*n, k_hinge = k_hinge, hinges = hinge_x_pairs, direction=0)
    stiffness_y = R_cx.apply_hinge_joints(N=49*n, k_hinge = k_hinge, hinges = hinge_y_pairs, direction=1)
    interconnection = stiffness_x + stiffness_y
    fem_k = fem_k + interconnection # Module stiffness matrix add hinge stiffness matrix
    # 主要控制节点

    # 总节点数
    num_nodes = 49 * n
    
    # 控制选项: 使用SEREP还是Static Condensation
    use_serep = False  # True表示使用SEREP，False表示使用Static Condensation

    return fem_k, fem_m, master_nodes, num_nodes, hydro_file, use_serep


# 计算程序
# 将单个结构的刚度矩阵拼接为大的矩阵
def create_block_matrix(matrix, n):
    """
    创建一个 n 个结构的块对角矩阵。
    matrix 是单个结构的刚度或质量矩阵，n 是结构的数量。
    """
    zero_matrix = np.zeros_like(matrix)
    blocks = []

    for i in range(n):
        row_blocks = [zero_matrix] * n
        row_blocks[i] = matrix
        blocks.append(row_blocks)

    big_matrix = np.block(blocks)
    return big_matrix


# 生成主控制点
def generate_control_points(structure_size=300, module_size=30):  
    """  
    生成结构中的100个控制点，节点编号按有限元编号方式累加  
    
    参数:  
    structure_size (float): 结构总尺寸 (m)  
    module_size (float): 模块尺寸 (m)  
    
    返回:  
    list: 包含所有控制点信息的列表  
    """  
    # 基本参数计算  
    num_modules_per_side = int(structure_size / module_size)  # 10  
    spacing = module_size  
    offset = spacing / 2  
    nodes_per_module = 49  # 每个模块的节点数  
    base_center_node = 25  # 第一个模块的中心节点号  
    
    # 生成x和y坐标  
    x_coords = np.linspace(offset, structure_size - offset, num_modules_per_side)  
    y_coords = np.linspace(structure_size - offset, offset, num_modules_per_side)  # 从上到下  
    
    # 存储控制点的列表  
    control_points = []  
    
    # 生成控制点  
    point_number = 1  
    for y in y_coords:  
        for x in x_coords:  
            # 计算当前模块的中心节点号  
            module_node_number = base_center_node + (point_number - 1) * nodes_per_module  
            
            control_point = {  
                'point_number': point_number,          # 控制点编号 (1-100)  
                'coordinates': (x, y),                 # 全局坐标  
                'fem_node_number': module_node_number, # 有限元节点编号  
                'module_position': (                   # 模块位置 (行,列)  
                    int((structure_size - y) / module_size) + 1,  
                    int(x / module_size) + 1  
                )  
            }  
            control_points.append(control_point)  
            point_number += 1  
    

def perform_calculation(fem_k, fem_m, master_nodes, num_nodes, hydro_file, use_serep):
    # 应用所有铰链关节到全局刚度矩阵
    # fem_kiffness = apply_hinge_joints(fem_k, k_hinge, hinges)
    # alpha = 1
    # fem_kiffness += alpha * np.eye(fem_kiffness.shape[0])

    cond_number = np.linalg.cond(fem_k)
    print(f"条件数: {cond_number}")

    # 使用 SEREP 或 static_condensation 进行降阶
    fem_m_reduced = SEREP.reduce_dofs(fem_m, num_nodes, [5])
    fem_siffness_reduced = SEREP.reduce_dofs(fem_k, num_nodes, [5])

    # 获取主自由度和从自由度
    MasterDofs, SlaveDofs = SEREP.separate_dofs(num_nodes, master_nodes)

    if use_serep:
        # 使用 SEREP 方法
        MR, KR, T = SEREP.SEREP(fem_siffness_reduced, fem_m_reduced, SlaveDofs, master_nodes)
    else:
        # 使用 Static Condensation 方法
        MR, KR, T = SEREP.static_condensation(fem_siffness_reduced, fem_m_reduced, MasterDofs, SlaveDofs)

    # 读取水动力数据
    dataset = merge_complex_values(xr.open_dataset(hydro_file))
    omega = dataset.omega.values
    i = 0

    # 获取水动力数据
    added_mass = dataset['added_mass'][i].values
    radiation_damping = dataset['radiation_damping'][i].values
    F_w = dataset['Froude_Krylov_force'][i].values + dataset['diffraction_force'][i].values

    # 对水动力矩阵进行降阶
    inertia_matrix = SEREP.reduce_dofs(dataset['inertia_matrix'].values, len(master_nodes), [5])
    added_mass = SEREP.reduce_dofs(added_mass, len(master_nodes), [5])
    radiation_damping = SEREP.reduce_dofs(radiation_damping, len(master_nodes), [5])
   
    # Capytaine 的净水刚度矩阵
    hydrostatic_stiffness = dataset['hydrostatic_stiffness'].values
    
    #######################定义自己算净水刚度###################################################
    #K_static_sub = SEREP.calculate_hydrostatic_stiffness_matrix(30, 60, 1.1, density=1000, gravity=9.80665)
    #hydrostatic_stiffness = block_diag(*[K_static_sub for _ in range(10)])
    ######################################定义弹簧刚度#########################################


    hydrostatic_stiffness = SEREP.reduce_dofs(hydrostatic_stiffness, len(master_nodes), [5])

    F_w = SEREP.reduce_force_matrix_dofs(F_w, len(master_nodes), 5)[::-1].reshape(1, 5 * len(master_nodes))

    # 生成系统矩阵
    mass = added_mass + MR # inertia_matrix
    damping = radiation_damping
    stiffness = hydrostatic_stiffness/1.05+ KR

    # 频域求解
    master_displacement = DM_A.solve_frequency_domain(mass, damping, stiffness, F_w, omega[i])
    global_displacement_disorder = T @ master_displacement
    master_displacement = master_displacement.reshape(len(master_nodes), 5)[::-1].reshape(5 * len(master_nodes), 1)

    # 计算全局位移矩阵
    global_displacement_error = SEREP.reorder_displacement_matrix(global_displacement_disorder, MasterDofs, SlaveDofs)
    global_displacement_replace = DM_M.replace_master_with_global(master_displacement, global_displacement_error, master_nodes)

    return global_displacement_replace


def post_process(global_displacement_replace): 
    from scipy.interpolate import interp1d
    import scienceplots
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    plt.style.use(['science', 'no-latex', 'ieee'])

    def calculate_rmse(model, reference):
        return np.sqrt(np.mean((model - reference) ** 2))

    def calculate_relative_error(model, reference):
        return np.abs((model - reference) / reference) * 100

    mid = global_displacement_replace[2::5, :]
    
    # 假设 heave 数据已经计算并存储在 heave 变量中
    heave = abs(mid)

    # 将 heave 数据分别划分为三个模块
    model1 = heave[0:273].reshape(13, 21)  # 每个模块13x21
    model2 = heave[273:546].reshape(13, 21)
    model3 = heave[546:819].reshape(13, 21)

    # 将三个模块的模型结合在一起
    combined_model = np.hstack((model1, model2, model3))

    # 删除每个模块之间的铰接部分（删除第20列和第41列）
    modified_model = np.delete(combined_model, [20, 41], axis=1)

    # 读取外部数据
    df1 = pd.read_csv(r'E:\phd\Code\DM-FEM2D\FEM_Reduce\Hinge\Yoon-2-hige180-180-3.csv', header=None)
    df2 = pd.read_csv(r'E:\phd\Code\DM-FEM2D\FEM_Reduce\Hinge\Yoon-2-hige180-180-2.csv', header=None)
    df3 = pd.read_csv(r'E:\phd\Code\DM-FEM2D\FEM_Reduce\Hinge\Yoon-2-hige180-180-1.csv', header=None)

    x = np.linspace(0, 1, 61)

    # 插值对齐函数
    def interpolate_reference(reference_x, reference_y, target_x):
        f_interp = interp1d(reference_x, reference_y, kind='linear', fill_value="extrapolate")
        return f_interp(target_x)

    # 插值数据
    df1_interp = interpolate_reference(df1.iloc[:, 0].values, df1.iloc[:, 1].values, x)
    df2_interp = interpolate_reference(df2.iloc[:, 0].values, df2.iloc[:, 1].values, x)
    df3_interp = interpolate_reference(df3.iloc[:, 0].values, df3.iloc[:, 1].values, x)

    # 第一行对比 (Case 1)
    rmse_case1 = calculate_rmse(modified_model[0, :][::-1], df1_interp)
    relative_error_case1 = calculate_relative_error(modified_model[0, :][::-1], df1_interp)
    mean_relative_error_case1 = np.mean(relative_error_case1)
    print(f"Case 1 - RMSE: {rmse_case1:.4f}, Mean Relative Error: {mean_relative_error_case1:.2f}%")

    plt.figure(figsize=(10, 2))
    plt.plot(x, df1_interp, label='Yoon et al.2014', linewidth=1.5)
    plt.plot(x, modified_model[0, :][::-1], label='RODM', linewidth=1.5)
    plt.xlabel('x/L')
    plt.ylabel('Heave Displacement (m)')
    plt.ylim(0, 2)
    plt.legend()
    plt.savefig('Case_1_Comparison.pdf', format='pdf')
    plt.close()

    # 中心行对比 (Case 2)
    df4 = pd.read_csv(r'E:\phd\Code\DM-FEM2D\FEM_Reduce\Hinge\Yoon_exp.csv', header=None)
    

    rmse_case2 = calculate_rmse(modified_model[7, :][::-1], df2_interp)
    relative_error_case2 = calculate_relative_error(modified_model[7, :][::-1], df2_interp)
    mean_relative_error_case2 = np.mean(relative_error_case2)
    print(f"Case 2 - RMSE: {rmse_case2:.4f}, Mean Relative Error: {mean_relative_error_case2:.2f}%")

    plt.figure(figsize=(10, 2))
    plt.plot(x, df2_interp, label='Yoon et al.2014', linewidth=1.5)
    plt.plot(x, modified_model[7, :][::-1], label='RODM', linewidth=1.5)
    plt.scatter(df4.iloc[:, 0], df4.iloc[:, 1], color='black', label='Experiment')
    plt.xlabel('x/L')
    plt.ylabel('Heave Displacement (m)')
    plt.ylim(0, 2)
    plt.legend()
    plt.savefig('Case_2_Comparison.pdf', format='pdf')
    plt.close()

    # 最后一行对比 (Case 3)
    rmse_case3 = calculate_rmse(modified_model[12, :][::-1], df3_interp)
    relative_error_case3 = calculate_relative_error(modified_model[12, :][::-1], df3_interp)
    mean_relative_error_case3 = np.mean(relative_error_case3)
    print(f"Case 3 - RMSE: {rmse_case3:.4f}, Mean Relative Error: {mean_relative_error_case3:.2f}%")

    plt.figure(figsize=(10, 2))
    plt.plot(x, df3_interp, label='Yoon et al.2014', linewidth=1.5)
    plt.plot(x, modified_model[12, :][::-1], label='RODM', linewidth=1.5)
    plt.xlabel('x/L')
    plt.ylabel('Heave Displacement (m)')
    plt.ylim(0, 2)
    plt.legend()
    plt.savefig('Case_3_Comparison.pdf', format='pdf')
    plt.close()

    return modified_model

def main():
    # 参数输入
    fem_k, fem_m, master_nodes, num_nodes, hydro_file, use_serep = input_parameters()
    
    # 计算程序
    #global_displacement_replace = perform_calculation(fem_k, fem_m, master_nodes, num_nodes, hydro_file, use_serep)
    
    # 后处理
    modified_model = post_process(global_displacement_replace)
    plt.imshow(modified_model,interpolation='spline16')

if __name__ == "__main__":
    main()


In [1]:
import numpy as np  

def generate_control_points(structure_size=300, module_size=30):  
    """  
    生成结构中的100个控制点，节点编号按有限元编号方式累加  
    
    参数:  
    structure_size (float): 结构总尺寸 (m)  
    module_size (float): 模块尺寸 (m)  
    
    返回:  
    list: 包含所有控制点信息的列表  
    """  
    # 基本参数计算  
    num_modules_per_side = int(structure_size / module_size)  # 10  
    spacing = module_size  
    offset = spacing / 2  
    nodes_per_module = 49  # 每个模块的节点数  
    base_center_node = 25  # 第一个模块的中心节点号  
    
    # 生成x和y坐标  
    x_coords = np.linspace(offset, structure_size - offset, num_modules_per_side)  
    y_coords = np.linspace(structure_size - offset, offset, num_modules_per_side)  # 从上到下  
    
    # 存储控制点的列表  
    control_points = []  
    
    # 生成控制点  
    point_number = 1  
    for y in y_coords:  
        for x in x_coords:  
            # 计算当前模块的中心节点号  
            module_node_number = base_center_node + (point_number - 1) * nodes_per_module  
            
            control_point = {  
                'point_number': point_number,          # 控制点编号 (1-100)  
                'coordinates': (x, y),                 # 全局坐标  
                'fem_node_number': module_node_number, # 有限元节点编号  
                'module_position': (                   # 模块位置 (行,列)  
                    int((structure_size - y) / module_size) + 1,  
                    int(x / module_size) + 1  
                )  
            }  
            control_points.append(control_point)  
            point_number += 1  
    
    return control_points  

# def print_control_points(points, num_to_show=None):  
#     """  
#     打印控制点信息  
    
#     参数:  
#     points (list): 控制点列表  
#     num_to_show (int): 要显示的点数量，None表示显示所有  
#     """  
#     print(f"\n总控制点数量: {len(points)}")  
#     print("\n控制点信息:")  
#     print("-" * 75)  
#     print("编号  |    X坐标    |    Y坐标    | 模块位置(行,列) | 有限元节点号")  
#     print("-" * 75)  
    
#     for point in points[:num_to_show]:  
#         print(f"{point['point_number']:3d}   | {point['coordinates'][0]:10.2f} | "  
#               f"{point['coordinates'][1]:10.2f} | ({point['module_position'][0]:2d},{point['module_position'][1]:2d})"  
#               f"           | {point['fem_node_number']}")  

# 使用示例  
# def main():  
#     # 生成控制点  
#     control_points = generate_control_points()  
#     print(len(control_points))
    
#     # 打印前20个控制点的信息  
#     print_control_points(control_points, num_to_show=20)  
    
#     # 返回完整列表供后续使用  
#     return control_points  

# if __name__ == "__main__":  
#     points = main()

In [2]:
points = generate_control_points()
master_nodes = []
for point in points:
    master_nodes.append(point['fem_node_number'])


In [3]:
len(master_nodes)

100